In [ ]:
from rocketpy import Environment, SolidMotor, Rocket, Flight, simulation
import datetime
import math

tomorrow = datetime.date.today() + datetime.timedelta(days=1)


**Create Environment**  

2 Ways of accomplishing the same thing  
Run either one of the next 2 cells  

In [ ]:
# Run either this cell or the next cell
# These 2 cells accomplish the same thing in different ways
# This cell type is set to "Code"

env = Environment(latitude=28.0, longitude=-82.0, elevation=10)
env.set_date((tomorrow.year, tomorrow.month, tomorrow.day, 12))  # Hour given in UTC time
env.set_atmospheric_model(type="Forecast", file="GFS")



**BUILDING THE MOTOR**

In [ ]:
M1939W = SolidMotor(
    thrust_source="AeroTech_M1939W.eng",
    dry_mass=3.33,  # kg
    dry_inertia=(0.148, 0.148, 0.00392),
    nozzle_radius=0.009325,  # m
    grain_number=4,
    grain_density=1709,  # kg/m3
    grain_outer_radius=0.0427355,  # m
    grain_initial_inner_radius=0.0142875,  # m
    grain_initial_height=0.1524,  # m
    grain_separation=0.01,  # m
    grains_center_of_mass_position=0.366,  # m
    center_of_dry_mass_position=0.366,  # m
    nozzle_position=0,
    burn_time=6.23,  # s
    coordinate_system_orientation="nozzle_to_combustion_chamber",
    )

**BUILDING THE ROCKET**

In [ ]:
I = (4.656,4.656,0.088)  # kg*m2

rocket = Rocket(
    radius=0.157/2, # m
    mass=18.3,  # kg
    inertia= I, #(4.656,4.656,0.088), kg*m2
    power_off_drag="Cd_vs_Mach_openrocket_clean.csv",
    power_on_drag="Cd_vs_Mach_openrocket_clean.csv",
    center_of_mass_without_motor=1.65,  # m
    coordinate_system_orientation="nose_to_tail",
    )

rocket.add_motor(M1939W,position=2.9)

**ADDING NOSE CONE**

In [ ]:
nose_cone=rocket.add_nose(
    length=0.762, kind="lvhaack", position=0.0
    )

**ADDING THE FINS**

In [ ]:
fin_set=rocket.add_trapezoidal_fins(
    n=4,
    root_chord=0.356,
    tip_chord=0.051,
    span=0.178,
    position=2.49,
    cant_angle=0,
    sweep_angle=55
    )

**ADDING PARCHUTES**

In [ ]:
main = rocket.add_parachute(
    name="main",
    cd_s=9.77,
    trigger=304.8,      # ejection altitude in meters
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
    )

drogue = rocket.add_parachute(
    name="drogue",
    cd_s=0.293,
    trigger="apogee",  # ejection at apogee
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
    )

**CREATING TEST LAUNCH**

In [ ]:
test_flight = Flight(
    rocket=rocket, environment=env, rail_length=5.18, inclination=83, heading=0
    )

**CALLING ON SIMULATION FUNCTIONS**

In [ ]:
#rocket.all_info()
#rocket.plots.static_margin()
rocket.draw()
test_flight.plots.trajectory_3d()
test_flight.plots.linear_kinematics_data()
test_flight.plots.energy_data()
test_flight.info()

**Prepare for Monte Carlo Simulations**

In [ ]:
from rocketpy import simulation, MonteCarlo
from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticFlight,
    StochasticNoseCone,
    StochasticParachute,
    #StochasticRailButtons,
    StochasticRocket,
    StochasticSolidMotor,
    #StochasticTail,
    StochasticTrapezoidalFins,
    )

**Create "Stochastic" Version of Everything**

In [ ]:
stochastic_env = StochasticEnvironment(env)
stochastic_env.visualize_attributes()

In [ ]:
stochastic_env = StochasticEnvironment(env)
stochastic_env.visualize_attributes()

stochastic_rocket = StochasticRocket(rocket)
stochastic_rocket.visualize_attributes()

stochastic_motor = StochasticSolidMotor(
    solid_motor=M1939W,
    burn_start_time=(0, 0.1, "binomial"),
    grains_center_of_mass_position=0.001,
    grain_density=10,
    grain_separation=1 / 1000,
    grain_initial_height=1 / 1000,
    grain_initial_inner_radius=0.375 / 1000,
    grain_outer_radius=0.375 / 1000,
    total_impulse=(6500, 100),
    throat_radius=0.5 / 1000,
    nozzle_radius=0.5 / 1000,
    nozzle_position=0.001,
    )
stochastic_motor.visualize_attributes()

stochastic_main_parachute = StochasticParachute(
    parachute=main,
    cd_s=0.1,
    lag=0.1,
    )
stochastic_main_parachute.visualize_attributes()

stochastic_drogue_parachute = StochasticParachute(
    parachute=drogue,
    cd_s=0.07,
    lag=0.2,
    )
stochastic_drogue_parachute.visualize_attributes()

stochastic_fins = StochasticTrapezoidalFins(
    trapezoidal_fins=fin_set,
    root_chord=0.0005,
    tip_chord=0.0005,
    span=0.0005,
    )
stochastic_fins.visualize_attributes()

stochastic_nose_cone = StochasticNoseCone(
    nosecone=nose_cone,
    length=0.001,
    )
stochastic_nose_cone.visualize_attributes()

stochastic_rocket.add_motor(stochastic_motor, position=0.001)
stochastic_rocket.add_nose(stochastic_nose_cone, position=(1.134, 0.001))
stochastic_rocket.add_trapezoidal_fins(stochastic_fins, position=(0.001, "normal"))
stochastic_rocket.add_parachute(stochastic_main_parachute)
stochastic_rocket.add_parachute(stochastic_drogue_parachute)

stochastic_flight = StochasticFlight(
    flight=test_flight,
    inclination=(83, 1),  # mean= 83, std=1
    heading=(0, 2),  # mean= 0, std=2
    )
stochastic_flight.visualize_attributes()


**Create Conte Carlo Simulation**

In [ ]:
test_dispersion = MonteCarlo(
    filename="monte_carlo_analysis_outputs",
    environment=stochastic_env,
    rocket=stochastic_rocket,
    flight=stochastic_flight,
    )

test_dispersion.simulate(
    number_of_simulations=10,
    append=False,
    include_function_data=False,
    #parallel=True,
    #n_workers=4,
    )

test_dispersion.prints.all()